# ECE 567 Final Project — Heart Disease (Kaggle s6e2)
## Notebook 1: EDA + Preprocessing

Furkan Yakkan, AGU, ECE 567.

Competition: https://www.kaggle.com/competitions/playground-series-s6e2

### Setup: mount Drive and locate the CSVs

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ECE567_Final/playground-series-s6e2'

import os
for f in sorted(os.listdir(DATA_DIR)):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f'{f:30s} {size_mb:8.2f} MB')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 2540
np.random.seed(SEED)

heart_train = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
heart_test  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print('train shape:', heart_train.shape)
print('test shape :', heart_test.shape)
heart_train.head()

### Inspect dtypes and find string columns

The Kaggle CSV stores the target as `'Presence'`/`'Absence'` and may store other clinical features as strings rather than numeric codes.

In [ ]:
print('--- dtypes ---')
print(heart_train.dtypes)

object_cols = heart_train.select_dtypes(include='object').columns.tolist()
print(f'\n--- {len(object_cols)} string columns ---')
for c in object_cols:
    uniq = heart_train[c].unique()
    print(f'  {c:30s} {len(uniq)} unique  ->  {uniq[:10]}')

### Encode strings to numeric

Target gets an explicit `Presence -> 1, Absence -> 0` map. Other string columns get label-encoded (deterministic, sorted) so the mapping is reproducible across train and test.

In [ ]:
# explicit target mapping
TARGET = 'Heart Disease'
if heart_train[TARGET].dtype == object:
    heart_train[TARGET] = heart_train[TARGET].map({'Presence': 1, 'Absence': 0})
assert heart_train[TARGET].isin([0, 1]).all(), 'target has unexpected values'

# label-encode every remaining string column using train categories
feature_string_cols = [c for c in object_cols if c != TARGET]
encoders = {}
for c in feature_string_cols:
    cats = sorted(heart_train[c].dropna().unique())
    mapping = {v: i for i, v in enumerate(cats)}
    encoders[c] = mapping
    heart_train[c] = heart_train[c].map(mapping)
    heart_test[c]  = heart_test[c].map(mapping)
    print(f'  {c}: {mapping}')

print('\n--- post-encoding dtypes ---')
print(heart_train.dtypes)

### Schema and types

In [ ]:
heart_train.info()

In [ ]:
heart_train.describe().T

### Missing values

In [ ]:
missing_train = heart_train.isna().sum()
missing_test  = heart_test.isna().sum()
print('--- train missing ---')
print(missing_train[missing_train > 0] if missing_train.sum() else 'none')
print('\n--- test missing ---')
print(missing_test[missing_test > 0] if missing_test.sum() else 'none')

### Target balance

In [ ]:
target_counts = heart_train[TARGET].value_counts(normalize=True).sort_index()
print(target_counts)
target_counts.plot(kind='bar', figsize=(4,3))
plt.title('Heart Disease class balance (train)')
plt.xticks(rotation=0)
plt.show()

### Univariate distributions

In [ ]:
feature_cols = [c for c in heart_train.columns if c not in ('id', TARGET)]
fig, axes = plt.subplots(4, 4, figsize=(14, 10))
for ax, col in zip(axes.flat, feature_cols):
    heart_train[col].hist(bins=40, ax=ax)
    ax.set_title(col, fontsize=9)
for ax in axes.flat[len(feature_cols):]:
    ax.axis('off')
plt.tight_layout()
plt.show()

### Correlation with target

In [ ]:
corr_with_y = heart_train[feature_cols + [TARGET]].corr()[TARGET].drop(TARGET)
corr_with_y.sort_values().plot(kind='barh', figsize=(6,4))
plt.title('Pearson correlation with Heart Disease')
plt.tight_layout()
plt.show()

### Save a stratified train/val split for downstream notebooks

In [ ]:
from sklearn.model_selection import train_test_split

X = heart_train[feature_cols].values
y = heart_train[TARGET].values

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print('train:', X_tr.shape, 'val:', X_val.shape)
print('train pos rate:', round(y_tr.mean(), 4))
print('val   pos rate:', round(y_val.mean(), 4))

OUT_DIR = '/content/drive/MyDrive/ECE567_Final/splits'
os.makedirs(OUT_DIR, exist_ok=True)
np.savez(
    os.path.join(OUT_DIR, 'split_raw.npz'),
    X_tr=X_tr, X_val=X_val, y_tr=y_tr, y_val=y_val,
    feature_cols=np.array(feature_cols),
)

# also save the test set ready for inference (no labels)
X_test = heart_test[feature_cols].values
test_ids = heart_test['id'].values
np.savez(
    os.path.join(OUT_DIR, 'test_raw.npz'),
    X_test=X_test, test_ids=test_ids,
    feature_cols=np.array(feature_cols),
)

# persist the encoder mappings for reproducibility
import json
with open(os.path.join(OUT_DIR, 'encoders.json'), 'w') as fh:
    json.dump({k: {str(kk): vv for kk, vv in v.items()} for k, v in encoders.items()}, fh, indent=2)

print('\nsaved:')
for f in sorted(os.listdir(OUT_DIR)):
    print(' ', os.path.join(OUT_DIR, f))